In [4]:
##### Converts raw raster on population gender groups into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # % of population female

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [5]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# raster directory 
raster_dir = Path(f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1") 

# save paths
pixels = f"{cd}/Data/Clean/Predictors/Rasters/female_share.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/female_share.csv"

In [6]:
#### Create gender rasters 

# set file pattern
def make_pattern(code):
    return f"global_{code}_2020_CN_1km_R2025A_UA_v1.tif"

female_codes = ["f_00", "f_01", "f_05", "f_10", "f_15", "f_20", "f_25", "f_30", "f_35", "f_40", "f_45", "f_50", "f_55", "f_60", "f_65", "f_70", "f_75", "f_80", "f_85", "f_90"]

# define function to sum rasters
def sum_rasters(codes, out_path):
    total = None
    meta  = None

    for code in codes:
        path = raster_dir / make_pattern(code)
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            nodata = src.nodata
            if meta is None:
                meta = src.meta.copy()

            # Mask nodata
            arr[arr == nodata] = np.nan

            if total is None:
                total = np.zeros_like(arr)

            # Add, treating nan as 0 (absent age group contributes nothing)
            total = np.nansum(np.stack([total, arr]), axis=0)

    # Where ALL inputs were nan, set back to nan
    all_nan = np.all(
        np.stack([
            np.isnan(rasterio.open(raster_dir / make_pattern(c)).read(1).astype(np.float32))
            for c in codes
        ]),
        axis=0
    )
    total[all_nan] = np.nan

    # Save
    meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
    out_arr = total.copy()
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(out_arr, 1)

    print(f"Saved: {out_path}  |  sum range: {np.nanmin(total):.2f} – {np.nanmax(total):.2f}")

sum_rasters(female_codes,  raster_dir / "pop_female_2020.tif")

Saved: /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/pop_female_2020.tif  |  sum range: 0.00 – 54881.36


In [9]:
#### Run resampling for pixel scale 

# set paths
female = f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/pop_female_2020.tif"
total = f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/total_pop_2020.tif"

### PREP 
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()


### RESAMPLE 
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

fem  = resample_to_ref(female)
tot = resample_to_ref(total)

### CALCULATE 
# compute female share
with np.errstate(invalid="ignore", divide="ignore"):
    fem_share = np.where(
        (tot > 0) & ~np.isnan(tot) & ~np.isnan(fem),
        fem / tot,
        np.nan
    )

fem_share = np.clip(fem_share, 0, 1)

# save 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = fem_share.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

In [8]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_fem  = rasterstats.zonal_stats(gdf_proj, female, stats="sum", nodata=-9999)
zonal_tot = rasterstats.zonal_stats(gdf_proj, total,   stats="sum", nodata=-9999)

### compute share at vector level
fem_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_fem])
tot_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_tot])

with np.errstate(invalid="ignore", divide="ignore"):
    fem_share = np.where(
        (tot_sums > 0) & ~np.isnan(tot_sums) & ~np.isnan(fem_sums),
        fem_sums / tot_sums,
        np.nan
    )

result["female_share"] = np.clip(fem_share, 0, 1)

### save
result.to_csv(vectors, index=False)